# 06 · Does dimensionality reduce the remaining score error?

Hold selection fixed at **random**, and compare D ∈ {1, 2, 3, 5, 8} with the same estimators,
family splits, seeds and budgets as 05. Do not mix a dimensionality change with a new selector.

1. Training rows only: centred-response PCA scree plot and Bernoulli parallel analysis.
2. Validation: likelihood / calibration on hidden answers, and full-score error for each D.
3. Pick D by random + gp-IRT mean log validation MAE across k (ties prefer the smaller D).
4. Freeze that choice, then report the predeclared D sweep on test. Test does not choose D.

The null uses independent Bernoulli matrices with training item means and the same shape.
This diagnoses response structure; it does not prove extra dimensions improve score estimation.

In [1]:
import sys

sys.path.insert(0, "../scripts/irt")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from efficbench import ROOT, load
from protocol import fit_context, make_splits
from run_progression import run_mirt

load_data = load  # Swap the loader while keeping response / model / item column order aligned.
ks = [10, 20, 50, 100, 200, 500]
seeds = range(20)

Y, models, items = load_data()
Y = Y.astype(np.float32)
splits = make_splits(Y)
subjects, model_ids = items.subject.values, models.model_id.values
item_ids = items.item_id.values


out = ROOT / "results/irt/progression/mirt"

# Replace the fitter OR supply {D: [one adapter per split]} to reuse existing models.
build_context = fit_context
contexts_by_dim = None
dims = [1, 2, 3, 5, 8]

## Model plug-in point

These stages do not require our fitter. Set `contexts` (or `contexts_by_dim` in 06) to
**one training-only frozen model per family split**, in that split order. Or replace
`build_context(Y_train, subjects, n_dims=...)` with your teammate's fitter.

For fitted 1PL/2PL/MIRT parameters:

```python
from cabench.design import parameter_context
ctx = parameter_context(train_theta, item_alpha, item_beta, item_ids=item_ids)
# Optional: retain your teammate's ability estimator (responses contains ONLY selected answers).
ctx["infer"] = teammate_infer  # (responses, items, initial=None) -> (models, D)
```

`item_beta` is the intercept in `sigmoid(theta @ alpha.T - beta)`; if the fit uses
`sigmoid(a * (theta-b))`, pass `beta=a*b`. Use ones for Rasch discrimination.
For a different backend, use `model_context` with prediction / inference callbacks and fixed
loadings; non-logistic links also need their correct `information_weights` callback.
See `scripts/irt/README.md` for shapes and examples. Keep item IDs, latent coordinates,
training prior and response columns aligned. Use a distinct `out` directory per fitted model.
The experiment helpers perform no refit when supplied with contexts.

In [ ]:
chosen_dims, chosen_contexts = run_mirt(
    Y,
    splits,
    subjects,
    model_ids,
    out,
    dims=sorted(contexts_by_dim) if contexts_by_dim is not None else dims,
    context_factory=build_context,
    contexts_by_dim=contexts_by_dim,
    ks=ks,
    seeds=seeds,
    item_ids=item_ids,
)
print(f"Validation-selected D = {chosen_dims}")

## Training-only dimensionality

Mean explained variance and actual eigenvalues versus the 95th percentile of 100 Bernoulli null
matrices, computed separately on each training split.

In [ ]:
diagnostics = pd.read_csv(out / "dimensionality.csv")
curve = diagnostics.groupby("component")[["variance_explained", "eigenvalue", "null_95"]].mean()
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(curve.index, 100 * curve.variance_explained, "o-")
left.set(xlabel="component", ylabel="variance explained (%)", title="Training scree plot")
right.plot(curve.index, curve.eigenvalue, "o-", label="response matrix")
right.plot(curve.index, curve.null_95, "o--", label="Bernoulli null: 95th percentile")
right.set(xlabel="component", ylabel="eigenvalue", yscale="log", title="Parallel analysis")
right.legend()
plt.tight_layout()

## Hidden-answer prediction on validation models

Reveal the same random half of questions at each D, infer ability using only those answers,
then measure log loss, Brier score and ten-bin calibration ECE on the hidden half.
The item-mean baseline ignores the responding model. These checks are separate from score MAE.

In [ ]:
response = pd.read_csv(out / "response_validation.csv")
response.groupby(["n_dims", "predictor"])[["log_loss", "brier", "ece"]].mean()

## Validation score error and the chosen D

gp-IRT weights are fitted on validation, so its validation error is tuning error.
Test below supplies the performance comparison.

In [ ]:
validation = pd.read_csv(out / "validation_predictions.csv")
val_gp = validation[validation.estimator == "gp-IRT"]
display(100 * val_gp.groupby(["k", "n_dims"]).abs_error.mean().unstack("n_dims"))
pd.read_csv(out / "chosen_dimension.csv")

## Test: same random subsets, different D

Look especially at k = 200 and 500. A lower error floor would support the multidimensional
hypothesis; the code does not assume that outcome.

In [ ]:
test = pd.read_csv(out / "test.csv")
gp = test[test.estimator == "gp-IRT"]
curve = gp.groupby(["n_dims", "k"]).mae.mean().unstack("n_dims")
(100 * curve).plot(
    marker="o",
    logx=True,
    figsize=(8, 4.5),
    xlabel="k (answers observed)",
    ylabel="test MAE (pp)",
    title=f"Random + gp-IRT; frozen validation choice D={chosen_dims}",
)
plt.tight_layout()
display(100 * curve)

Paired split-bootstrap comparisons use random + gp-IRT at the smallest supplied D as the reference (D=1 by default).
Seeds are averaged before resampling split means; the family splits overlap.

In [ ]:
paired = pd.read_csv(out / "paired.csv")
paired = paired[paired.method == "random + gp-IRT"].copy()
paired[["delta_mae", "ci_low", "ci_high"]] *= 100
paired[
    ["n_dims", "k", "delta_mae", "ci_low", "ci_high", "splits_won", "n_splits", "reference_dims"]
]

Dimensionality diagnostics follow the motivation in
[Krumdick et al. (2026)](https://arxiv.org/abs/2604.01418).
The selected D is passed to 07; no multidimensional scalar Fisher ranking is introduced.